# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates the process of loading, exploring, and analyzing the [FAIR² protein mass spectrometry](https://doi.org/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset is defined by a Croissant schema accessible via the following URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and inspect the dataset's top-level information using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the Croissant Dataset
dataset = mlc.Dataset(croissant_url)

# Display the main metadata fields
metadata = dataset.metadata
print(f"Dataset Title: {getattr(metadata, 'name', None)}\n")
print(f"Description: {getattr(metadata, 'description', None)}\n")
print(f"Citation: {getattr(metadata, 'citeAs', None)}\n")
print(f"Published: {getattr(metadata, 'datePublished', None)}\n")

## 2. Data Overview
List available record sets and their fields by referencing their `@id` fields. 

Fields within the Croissant dataset are typically organized in record sets, each containing its fields and columns. Let's display all record sets in the dataset and the fields in each one.

In [ ]:
# Display all record sets and their field @ids for reference

record_set_objs = dataset.metadata.recordSet
record_set_ids = []
for rec in record_set_objs:
    rec_id = getattr(rec, '@id', None)
    record_set_ids.append(rec_id)
    print(f"RecordSet @id: {rec_id}")
    if hasattr(rec, 'field'):
        for field in rec.field:
            field_id = getattr(field, '@id', None)
            field_name = getattr(field, 'name', None)
            print(f"  - Field @id: {field_id}\tname: {field_name}")
    print()

# For simplicity, print list of all record set ids
print('All available RecordSet @id values:')
print(record_set_ids)

## 3. Data Extraction
Let's load the data from a specific record set as a DataFrame for analysis. Use the record set and field `@id`s as identified above.

In [ ]:
# Replace with the actual record set ids found in the dataset
# If there is only one main record set for the protein data, use that
main_record_set_id = record_set_ids[0]  # Change index/select as needed based on printed output above

print(f"Loading data from RecordSet: {main_record_set_id}\n")
records = list(dataset.records(record_set=main_record_set_id))
df = pd.DataFrame(records)
# If field @id's differ from column headers, print available columns
print('DataFrame columns:', df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
Demonstrates common EDA steps like filtering, normalization, and aggregation. Let's choose a numeric field (by `@id` or column name) for analysis.

In [ ]:
# Pick a representative numeric field (e.g., 'MW', 'peptideCount', etc.),
# Replace as needed after confirming field/column names with df.columns above.
possible_numeric_fields = [col for col in df.columns if df[col].dtype in ['int64', 'float64']] or df.select_dtypes(include=['number']).columns.tolist()
print('Detected numeric fields:', possible_numeric_fields)
# Use 'MW' (Molecular Weight) if present, else fallback to first numeric field
numeric_field = 'MW' if 'MW' in df.columns else possible_numeric_fields[0]

threshold = df[numeric_field].quantile(0.75)  # use 75th quantile as threshold for demonstration
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
print(filtered_df.head())

norm_field = f"{numeric_field}_normalized"
filtered_df[norm_field] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"\nNormalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, norm_field]].head())

# Try grouping by a likely categorical field (e.g., 'sampleID', 'experiment', etc.)
possible_group_fields = [col for col in df.columns if df[col].nunique() < 10 and col != numeric_field]
group_field = possible_group_fields[0] if possible_group_fields else None
if group_field:
    grouped = filtered_df.groupby(group_field)[numeric_field].mean()
    print(f"\nGrouped mean of {numeric_field} by {group_field}:")
    print(grouped)
else:
    print("No suitable group field found for aggregation.")

## 5. Visualization
Visualize distributions or relationships between two variables. For example, histogram of a numeric field, boxplot by group, or scatter plot for two numeric fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

# Histogram of the numeric field
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.show()

# Boxplot by group_field if applicable
if group_field:
    plt.figure(figsize=(8,4))
    sns.boxplot(x=group_field, y=numeric_field, data=df)
    plt.title(f"{numeric_field} by {group_field}")
    plt.show()

## 6. Conclusion
- We demonstrated how to load and explore the FAIR² mass spectrometry dataset with mlcroissant.
- Record sets and fields were referenced using their `@id`s.
- Simple data filtering, normalization, and visualization techniques were applied.

For further exploration, consult the [mlcroissant documentation](https://mlcommons.github.io/croissant/) and refer to field `@id`s printed above when filtering or aggregating on specific variables.